In [ ]:
# ==========================================
# 0. INSTALL MISSING PACKAGES
# ==========================================
!pip install -q iterative-stratification

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import gc
import random
import os
import shutil
import glob
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from sklearn.metrics import f1_score
import lightgbm as lgb
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding, set_seed
)

# ==========================================
# 1. DISK CLEANUP & CONFIGURATION
# ==========================================
print("Cleaning up old model files from disk...")
for folder in glob.glob('./model_output*'):
    shutil.rmtree(folder, ignore_errors=True)

labels = ['E', 'S', 'G', 'non_ESG']

# The Exact 3 Models that generated the 0.92 Score
models_to_train = [
    "climatebert/distilroberta-base-climate-f",
    "distilbert-base-uncased",
    "microsoft/deberta-v3-base"
]

# We only need Seed 42 because the interruption deleted the rest!
seed = 42
n_splits = 7  # 7-fold CV as per the log
device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 2. SEEDING & DATA LOADING
# ==========================================
def seed_everything(s):
    random.seed(s)
    os.environ['PYTHONHASHSEED'] = str(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    set_seed(s)

seed_everything(seed)

train_df = pd.read_csv('/kaggle/input/datasets/kabilgannouni/godatasc/train.csv')
test_df = pd.read_csv('/kaggle/input/datasets/kabilgannouni/godatasc/test.csv')

train_df = train_df.dropna(subset=['text']).reset_index(drop=True)
test_texts = test_df['text'].fillna("").values
train_df[labels] = train_df[labels].astype(float)

# ==========================================
# 3. HELPER FUNCTIONS
# ==========================================
class ESGWeightedTrainer(Trainer):
    def __init__(self, pos_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weights = pos_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits if hasattr(outputs, "logits") else outputs
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def combine_labels(batch):
    batch['labels'] = [[batch[l][i] for l in labels] for i in range(len(batch[labels[0]]))]
    return batch

# ==========================================
# 4. TRAINING LOOP (ONLY SEED 42)
# ==========================================
train_oof_probs = np.zeros((len(train_df), len(labels), len(models_to_train)))
test_model_probs = np.zeros((len(test_df), len(labels), len(models_to_train)))

mskf = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

for model_idx, model_nm in enumerate(models_to_train):
    print(f"\n{'*'*40}")
    print(f" TRAINING MODEL: {model_nm}")
    print(f"{'*'*40}\n")
   
    tokenizer = AutoTokenizer.from_pretrained(model_nm)
   
    def tokenize_func(examples):
        return tokenizer(examples["text"], truncation=True, max_length=128)
       
    ds_test = Dataset.from_pandas(pd.DataFrame({'text': test_texts})).map(tokenize_func, batched=True)
   
    model_test_preds_accum = np.zeros((len(test_df), len(labels)))
    targets = train_df[labels].values
   
    for fold, (train_idx, val_idx) in enumerate(mskf.split(train_df, targets)):
        print(f"\n--- FOLD {fold+1}/{n_splits} ---")
       
        train_data = train_df.iloc[train_idx].reset_index(drop=True)
        val_data = train_df.iloc[val_idx].reset_index(drop=True)
       
        counts = train_data[labels].sum().values
        weights = torch.tensor([len(train_data) / (len(labels) * (c + 1e-5)) for c in counts], dtype=torch.float).to(device)
       
        ds_train = Dataset.from_pandas(train_data).map(tokenize_func, batched=True).map(combine_labels, batched=True)
        ds_val = Dataset.from_pandas(val_data).map(tokenize_func, batched=True).map(combine_labels, batched=True)
       
        model = AutoModelForSequenceClassification.from_pretrained(
            model_nm, num_labels=len(labels), problem_type="multi_label_classification"
        )
       
        # Exact learning rate and batch sizes from teammate's logic
        lr = 3e-5 if "roberta" in model_nm or "deberta" in model_nm else 5e-5
        batch_size = 16 if "large" not in model_nm else 8
       
        args = TrainingArguments(
            output_dir=f'model_output_fold{fold}',
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            learning_rate=lr,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=32,
            num_train_epochs=3,
            weight_decay=0.01,
            warmup_ratio=0.1,
            lr_scheduler_type="cosine",
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            fp16=True,
            report_to="none"
        )
       
        data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
        trainer = ESGWeightedTrainer(
            pos_weights=weights, model=model, args=args, train_dataset=ds_train,
            eval_dataset=ds_val, tokenizer=tokenizer, data_collator=data_collator
        )
       
        trainer.train()
       
        # OOF Predictions
        val_preds_raw = trainer.predict(ds_val)
        val_probs = 1 / (1 + np.exp(-val_preds_raw.predictions))
        train_oof_probs[val_idx, :, model_idx] = val_probs
       
        # Test Predictions
        test_preds_raw = trainer.predict(ds_test)
        test_probs = 1 / (1 + np.exp(-test_preds_raw.predictions))
        model_test_preds_accum += test_probs / n_splits
       
        # Cleanup
        del model, trainer, ds_train, ds_val
        torch.cuda.empty_cache()
        gc.collect()
        shutil.rmtree(args.output_dir, ignore_errors=True)

    test_model_probs[:, :, model_idx] = model_test_preds_accum

# ==========================================
# 5. THE WINNING LIGHTGBM META-LEARNER
# ==========================================
print("\n" + "="*45)
print("TRAINING LIGHTGBM META-LEARNER ON 3-MODEL OOF PROBS...")

final_stacked_preds = np.zeros((len(test_df), len(labels)))

for class_idx, label in enumerate(labels):
    print(f"Fitting Meta-Learner for class: {label}")
   
    X_train = train_oof_probs[:, class_idx, :]
    y_train = train_df[label].values
    X_test = test_model_probs[:, class_idx, :]
   
    # Exact params from the emergency script
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'random_state': 777,
        'learning_rate': 0.05,
        'is_unbalance': True
    }
   
    dtrain = lgb.Dataset(X_train, label=y_train)
    meta_model = lgb.train(params, dtrain, num_boost_round=100)
   
    y_pred_proba = meta_model.predict(X_test)
   
    # EXACT THRESHOLD FROM EMERGENCY SCRIPT (0.48)
    final_stacked_preds[:, class_idx] = (y_pred_proba >= 0.48).astype(int)

# ==========================================
# 6. POST-PROCESSING (ESG Logic Recovery)
# ==========================================
print("Applying Logical ESG Post-Processing...")
# Logical correction: If E, S, or G is tagged, it cannot be 'non_ESG'
# If nothing is tagged, it MUST be 'non_ESG'
for i in range(len(final_stacked_preds)):
    has_esg = np.any(final_stacked_preds[i, :3] == 1)
    if has_esg:
        final_stacked_preds[i, 3] = 0
    else:
        final_stacked_preds[i, 3] = 1

# ==========================================
# 7. FINAL SUBMISSION EXPORT
# ==========================================
submission_df = pd.DataFrame(final_stacked_preds.astype(int), columns=labels)
submission_df.insert(0, 'id', test_df['id'].values)
submission_df.to_csv("esg_rank1_phase2_final.csv", index=False)
print("\n SUCCESS! Clean Phase 2 Submission saved: esg_rank1_phase2_final.csv")